<a href="https://colab.research.google.com/github/BozpolatQTG/gulsen_unuvar_bitirme_projesi-/blob/main/_kuantum_mant%C4%B1k_kap%C4%B1lar%C4%B1_demo_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## kubit sayısı azaltılmış farklı simülatörlü kod

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from itertools import product


def full_adder_inplace(circ, a, b, cin, cout):
    circ.ccx(a, b, cout)
    circ.cx(a, b)
    circ.ccx(cin, b, cout)
    circ.cx(cin, b)


def make_registers(n: int):
    A = QuantumRegister(n, "A")
    B = QuantumRegister(n, "B")
    K = QuantumRegister(1, "K")
    C = QuantumRegister(n, "C")
    cl = ClassicalRegister(n + 1, "cl")
    return A, B, K, C, cl


def load_inputs(qc, A, B, A_val, B_val, n):
    for i in range(n):
        if (A_val >> i) & 1:
            qc.x(A[i])
        if (B_val >> i) & 1:
            qc.x(B[i])


def apply_mode_on_B(qc, B, K, n):
    for i in range(n):
        qc.cx(K[0], B[i])


def ripple_addsub_inplace(qc, A, B, K, C, n):
    full_adder_inplace(qc, A[0], B[0], K[0], C[0])

    for i in range(1, n):
        full_adder_inplace(qc, A[i], B[i], C[i - 1], C[i])


def add_measurements(qc, B, C, cl, n):
    for i in range(n):
        qc.measure(B[i], cl[i])

    qc.measure(C[n - 1], cl[n])


def twos_complement_to_int(bitstring: str) -> int:
    width = len(bitstring)
    value = int(bitstring, 2)

    if bitstring[0] == "1":
        value -= (1 << width)

    return value


def trim_binary(bitstring: str) -> str:
    trimmed = bitstring.lstrip("0")
    return trimmed if trimmed != "" else "0"


def run_one(A_val, B_val, K_val, n, backend, shots=1024):
    if A_val < 0 or B_val < 0:
        raise ValueError("A_val ve B_val negatif olamaz.")

    if A_val >= (1 << n) or B_val >= (1 << n):
        raise ValueError(f"A veya B, seçilen n={n} bit içine sığmıyor.")

    A, B, K, C, cl = make_registers(n)
    qc = QuantumCircuit(A, B, K, C, cl)

    load_inputs(qc, A, B, A_val, B_val, n)

    if K_val == 1:
        qc.x(K[0])

    apply_mode_on_B(qc, B, K, n)
    ripple_addsub_inplace(qc, A, B, K, C, n)
    add_measurements(qc, B, C, cl, n)

    compiled = transpile(qc, backend=backend, optimization_level=0)
    result = backend.run(compiled, shots=shots).result()
    counts = result.get_counts(compiled)
    bitstr = max(counts, key=counts.get)

    bits_lsb = bitstr[::-1]
    sum_lsb = bits_lsb[0:n]
    c_last = bits_lsb[n]

    sum_bin = "".join(sum_lsb[::-1])
    out_bin = c_last + sum_bin

    if K_val == 0:
        out_dec = int(out_bin, 2)
        mode = "ADD"
    else:
        out_dec = twos_complement_to_int(sum_bin)
        mode = "SUB"

    return {
        "mode": mode,
        "A_dec": A_val,
        "B_dec": B_val,
        "A_bin": trim_binary(format(A_val, f"0{n}b")),
        "B_bin": trim_binary(format(B_val, f"0{n}b")),
        "sum_bin": trim_binary(sum_bin),
        "carry_out": c_last,
        "out_bin": trim_binary(out_bin),
        "out_dec": out_dec,
        "circuit": qc
    }


def print_table(n=10, limit=11, shots=1024):
    backend = AerSimulator(method="matrix_product_state")

    if limit > (1 << n):
        raise ValueError(f"limit={limit}, seçilen n={n} bit için çok büyük.")

    for K_val, mode in [(0, "ADD"), (1, "SUB")]:
        print(f"\nMode = {mode} (K={K_val})")
        print(" A(dec)| B(dec)| S(dec) || A(bin) | B(bin) | S(bin)")
        print("-" * 82)

        for A_val, B_val in product(range(limit), range(limit)):
            data = run_one(A_val, B_val, K_val, n, backend, shots)

            print(
                f" {data['A_dec']:5d} |"
                f" {data['B_dec']:5d} |"
                f" {data['out_dec']:11d} || "
                f"{data['A_bin']} | {data['B_bin']} | {data['out_bin']}"
            )


if __name__ == "__main__":
    print_table(n=10, limit=11, shots=1024)


Mode = ADD (K=0)
 A(dec)| B(dec)| S(dec) || A(bin) | B(bin) | S(bin)
----------------------------------------------------------------------------------
     0 |     0 |           0 || 0 | 0 | 0
     0 |     1 |           1 || 0 | 1 | 1
     0 |     2 |           2 || 0 | 10 | 10
     0 |     3 |           3 || 0 | 11 | 11
     0 |     4 |           4 || 0 | 100 | 100
     0 |     5 |           5 || 0 | 101 | 101
     0 |     6 |           6 || 0 | 110 | 110
     0 |     7 |           7 || 0 | 111 | 111
     0 |     8 |           8 || 0 | 1000 | 1000
     0 |     9 |           9 || 0 | 1001 | 1001
     0 |    10 |          10 || 0 | 1010 | 1010
     1 |     0 |           1 || 1 | 0 | 1
     1 |     1 |           2 || 1 | 1 | 10
     1 |     2 |           3 || 1 | 10 | 11
     1 |     3 |           4 || 1 | 11 | 100
     1 |     4 |           5 || 1 | 100 | 101
     1 |     5 |           6 || 1 | 101 | 110
     1 |     6 |           7 || 1 | 110 | 111
     1 |     7 |           8 || 1 |